In [0]:
CREATE OR REPLACE TABLE f1.gold.dim_driver AS

WITH appearances AS (
  SELECT d.driver_id, d.first_name, d.last_name, d.name_acronym, d.country_code,
         d.team_name, d.driver_number, s.date_start
  FROM f1.silver.session_drivers d
  JOIN f1.silver.sessions s ON s.session_key = d.session_key
  WHERE s.session_type = 'Race' AND s.status = 'completed'
),

changes AS (
  SELECT *,
    CASE WHEN team_name     = LAG(team_name)     OVER w
          AND driver_number = LAG(driver_number) OVER w
         THEN 0 ELSE 1 END AS is_change
  FROM appearances
  WINDOW w AS (PARTITION BY driver_id ORDER BY date_start)
),

versions AS (
  SELECT *,
    SUM(is_change) OVER (PARTITION BY driver_id ORDER BY date_start) AS version_no
  FROM changes
),

spans AS (
  -- Type 2: one row per period of the same team and number
  SELECT driver_id, version_no, team_name, driver_number,
         MIN(date_start) AS valid_from
  FROM versions
  GROUP BY driver_id, version_no, team_name, driver_number
),

latest_details AS (
  -- Type 1: name and nationality always show the most recent known value
  SELECT driver_id,
         MAX_BY(first_name,   date_start) FILTER (WHERE first_name   IS NOT NULL) AS first_name,
         MAX_BY(last_name,    date_start) FILTER (WHERE last_name    IS NOT NULL) AS last_name,
         MAX_BY(name_acronym, date_start) FILTER (WHERE name_acronym IS NOT NULL) AS name_acronym,
         MAX_BY(country_code, date_start) FILTER (WHERE country_code IS NOT NULL) AS country_code
  FROM appearances
  GROUP BY driver_id
)

SELECT
  concat(s.driver_id, '_v', s.version_no)  AS driver_key,
  s.driver_id,
  s.version_no,
  d.first_name, d.last_name, d.name_acronym, d.country_code,
  s.team_name,
  s.driver_number,
  s.valid_from,
  LEAD(s.valid_from) OVER (PARTITION BY s.driver_id ORDER BY s.version_no)            AS valid_to,
  LEAD(s.valid_from) OVER (PARTITION BY s.driver_id ORDER BY s.version_no) IS NULL    AS is_current
FROM spans s
JOIN latest_details d ON d.driver_id = s.driver_id

In [0]:
-- 1. Follow a few drivers through their careers
SELECT driver_id, version_no, team_name, driver_number, valid_from, valid_to, is_current
FROM f1.gold.dim_driver
WHERE driver_id IN ('liam_lawson', 'oliver_bearman', 'lando_norris', 'max_verstappen')
ORDER BY driver_id, version_no

In [0]:
-- 2. Every driver must have exactly one current row (expect no rows)
SELECT driver_id, COUNT_IF(is_current) AS current_rows
FROM f1.gold.dim_driver
GROUP BY driver_id
HAVING current_rows <> 1

In [0]:
-- 3. driver_key must be unique (expect no rows)
SELECT driver_key, COUNT(*) AS rows
FROM f1.gold.dim_driver
GROUP BY driver_key
HAVING rows > 1

In [0]:
CREATE OR REPLACE TABLE f1.gold.dim_session AS

WITH rounds AS (
  -- Official round numbers: count Grand Prix weekends in date order,
  -- skipping testing and cancelled weekends (F1 renumbers around them)
  SELECT meeting_key,
         DENSE_RANK() OVER (PARTITION BY year ORDER BY date_start) AS round_number
  FROM f1.silver.meetings
  WHERE NOT is_testing AND race_status <> 'cancelled'
)

SELECT
  s.session_key,
  s.meeting_key,
  s.year,
  r.round_number,
  m.meeting_name,                 -- e.g. "Australian Grand Prix"
  s.session_name,                 -- Race, Sprint, Qualifying, ...
  s.session_type,
  s.date_start,
  s.location,
  s.country_name,
  s.circuit_short_name,
  s.status
FROM f1.silver.sessions s
JOIN f1.silver.meetings m ON m.meeting_key = s.meeting_key
LEFT JOIN rounds r        ON r.meeting_key = s.meeting_key

In [0]:
CREATE OR REPLACE TABLE f1.gold.fact_session_result AS

SELECT
  r.session_key,
  dd.driver_key,                  -- the driver's team and number AT THAT RACE
  sd.driver_id,                   -- the person, for career totals
  r.status,
  r.position,
  r.finish_order,
  g.grid_position,
  g.grid_position - r.position    AS positions_gained,   -- null if not classified or no grid
  r.points,
  r.laps_completed,
  r.race_time_sec,
  r.gap_to_leader_sec,
  r.laps_behind
FROM f1.silver.session_results r
JOIN f1.silver.sessions s
  ON s.session_key = r.session_key
JOIN f1.silver.session_drivers sd
  ON  sd.session_key   = r.session_key
  AND sd.driver_number = r.driver_number
-- SCD Type 2 lookup: the driver period that was valid on race day
JOIN f1.gold.dim_driver dd
  ON  dd.driver_id = sd.driver_id
  AND s.date_start >= dd.valid_from
  AND (dd.valid_to IS NULL OR s.date_start < dd.valid_to)
LEFT JOIN f1.silver.starting_grid g
  ON  g.session_key   = r.session_key
  AND g.driver_number = r.driver_number

In [0]:
-- 1. Same row count as silver results (2,262), and every row found its driver period
SELECT COUNT(*) AS rows, COUNT_IF(driver_key IS NULL) AS no_driver_key
FROM f1.gold.fact_session_result

In [0]:
-- 2. Reality check: 2025 championship top 5, races plus sprints
SELECT f.driver_id, SUM(f.points) AS points
FROM f1.gold.fact_session_result f
JOIN f1.gold.dim_session ds ON ds.session_key = f.session_key
WHERE ds.year = 2025
GROUP BY f.driver_id
ORDER BY points DESC
LIMIT 5

In [0]:
CREATE OR REPLACE TABLE f1.gold.fact_lap AS

SELECT
  l.session_key,
  dd.driver_key,
  sd.driver_id,
  l.lap_number,
  l.lap_start,
  l.lap_time_sec,
  l.sector_1_sec,
  l.sector_2_sec,
  l.sector_3_sec,
  l.speed_trap_kph,
  l.is_pit_out_lap,
  l.is_clean_lap,
  st.stint_number,
  st.compound,
  st.compound_type,
  -- Tyre age on this lap: age when fitted + laps done on it so far
  st.tyre_age_start + (l.lap_number - st.lap_start)   AS tyre_age_laps
FROM f1.silver.laps l
JOIN f1.silver.sessions s
  ON s.session_key = l.session_key
JOIN f1.silver.session_drivers sd
  ON  sd.session_key   = l.session_key
  AND sd.driver_number = l.driver_number
JOIN f1.gold.dim_driver dd
  ON  dd.driver_id = sd.driver_id
  AND s.date_start >= dd.valid_from
  AND (dd.valid_to IS NULL OR s.date_start < dd.valid_to)
-- Range join: the stint whose lap range contains this lap
LEFT JOIN f1.silver.stints st
  ON  st.session_key   = l.session_key
  AND st.driver_number = l.driver_number
  AND l.lap_number BETWEEN st.lap_start AND st.lap_end

In [0]:
-- 1. No laps lost, none duplicated (expect 106,254 and 0)
SELECT COUNT(*) AS rows,
       COUNT(*) - COUNT(DISTINCT session_key, driver_key, lap_number) AS duplicates
FROM f1.gold.fact_lap

In [0]:
-- 2. How many laps couldn't be matched to a tyre?
SELECT COUNT_IF(compound IS NULL)                        AS laps_without_tyre,
       ROUND(COUNT_IF(compound IS NULL) / COUNT(*) * 100, 2) AS pct
FROM f1.gold.fact_lap

In [0]:
-- 3. Plausibility: tyre age should be 0 or more, and rarely above ~50 laps
SELECT MIN(tyre_age_laps) AS min_age, MAX(tyre_age_laps) AS max_age
FROM f1.gold.fact_lap

In [0]:
SELECT a.session_key, a.driver_number,
       a.stint_number AS stint_a, a.lap_start AS a_start, a.lap_end AS a_end, a.compound AS a_compound,
       b.stint_number AS stint_b, b.lap_start AS b_start, b.lap_end AS b_end, b.compound AS b_compound
FROM f1.silver.stints a
JOIN f1.silver.stints b
  ON  a.session_key   = b.session_key
  AND a.driver_number = b.driver_number
  AND a.stint_number  < b.stint_number      -- each pair once
  AND a.lap_start    <= b.lap_end           -- the two ranges overlap
  AND b.lap_start    <= a.lap_end
ORDER BY a.session_key, a.driver_number

In [0]:
CREATE OR REPLACE TABLE f1.gold.agg_tyre_degradation AS

WITH wet_sessions AS (
  -- Sessions where anyone used intermediate or wet tyres = mixed conditions
  SELECT DISTINCT session_key
  FROM f1.gold.fact_lap
  WHERE compound_type = 'wet'
),

stint_slopes AS (
  SELECT
    f.session_key,
    f.driver_id,
    f.stint_number,
    f.compound,
    COUNT(*)                                     AS clean_laps,
    REGR_SLOPE(f.lap_time_sec, f.tyre_age_laps)  AS sec_lost_per_lap
  FROM f1.gold.fact_lap f
  WHERE f.is_clean_lap
    AND f.compound IN ('SOFT', 'MEDIUM', 'HARD')
  GROUP BY f.session_key, f.driver_id, f.stint_number, f.compound
  HAVING clean_laps >= 8
)

SELECT
  ds.year,
  ds.round_number,
  ds.meeting_name,
  ds.session_name,
  ss.compound,
  w.session_key IS NOT NULL                         AS mixed_conditions,
  COUNT(*)                                          AS stints,
  ROUND(PERCENTILE(ss.sec_lost_per_lap, 0.5), 3)    AS median_sec_lost_per_lap
FROM stint_slopes ss
JOIN f1.gold.dim_session ds ON ds.session_key = ss.session_key
LEFT JOIN wet_sessions w    ON w.session_key  = ss.session_key
GROUP BY ds.year, ds.round_number, ds.meeting_name, ds.session_name,
         ss.compound, w.session_key

In [0]:
-- 1. Overall: softer tyres should degrade faster (SOFT > MEDIUM > HARD)
SELECT compound,
       SUM(stints)                              AS stints,
       ROUND(AVG(median_sec_lost_per_lap), 3)   AS avg_sec_lost_per_lap
FROM f1.gold.agg_tyre_degradation
GROUP BY compound
ORDER BY avg_sec_lost_per_lap DESC

In [0]:
SELECT year, meeting_name, session_name, compound, stints, median_sec_lost_per_lap
FROM f1.gold.agg_tyre_degradation
WHERE stints >= 5 AND NOT mixed_conditions
ORDER BY median_sec_lost_per_lap DESC
LIMIT 10

In [0]:
SELECT mixed_conditions, COUNT(*) AS rows
FROM f1.gold.agg_tyre_degradation
GROUP BY mixed_conditions

In [0]:
CREATE OR REPLACE TABLE f1.gold.agg_championship_progress AS

WITH round_points AS (
  -- Points per driver per weekend (race + sprint together)
  SELECT ds.year, ds.round_number, f.driver_id, SUM(f.points) AS points
  FROM f1.gold.fact_session_result f
  JOIN f1.gold.dim_session ds ON ds.session_key = f.session_key
  GROUP BY ds.year, ds.round_number, f.driver_id
),

rounds AS (
  SELECT DISTINCT year, round_number, meeting_name
  FROM f1.gold.dim_session
  WHERE session_name = 'Race' AND status = 'completed'
),

drivers AS (
  SELECT DISTINCT year, driver_id FROM round_points
),

full_grid AS (
  -- Every driver x every round of their season
  SELECT r.year, r.round_number, r.meeting_name, d.driver_id
  FROM rounds r
  JOIN drivers d ON d.year = r.year
),

filled AS (
  SELECT g.*, COALESCE(p.points, 0) AS round_points
  FROM full_grid g
  LEFT JOIN round_points p
    ON  p.year = g.year
    AND p.round_number = g.round_number
    AND p.driver_id = g.driver_id
),

cumulative AS (
  SELECT *,
    SUM(round_points) OVER (PARTITION BY year, driver_id ORDER BY round_number) AS total_points
  FROM filled
)

SELECT *,
  RANK() OVER (PARTITION BY year, round_number ORDER BY total_points DESC) AS championship_position
FROM cumulative

In [0]:
-- 1. Final 2025 standings should match the championship: 423 / 421 / 410
SELECT driver_id, total_points, championship_position
FROM f1.gold.agg_championship_progress
WHERE year = 2025
  AND round_number = (SELECT MAX(round_number) FROM f1.gold.agg_championship_progress WHERE year = 2025)
ORDER BY championship_position
LIMIT 5

In [0]:
-- 2. Who led after each round of 2025?
SELECT round_number, meeting_name, driver_id, total_points
FROM f1.gold.agg_championship_progress
WHERE year = 2025 AND championship_position = 1
ORDER BY round_number